In [1]:
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv
import os
import time

# === Load GitHub Token ===
load_dotenv('All_tokens.env')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

# === Paths ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
output_csv = Path(r"C:\GitHub\AndroidProjects\8.2-Project_Metadata.csv")

# === Load and clean URLs ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]

# === Extract metadata from API ===
project_metadata = []

for i, url in enumerate(df['github_url'], 1):
    parts = url.strip().split('/')
    if len(parts) < 5:
        continue

    username, repo = parts[-2], parts[-1].replace('.git', '')
    repo_name = f"{username}.{repo}"
    print(f"🔍 [{i}/{len(df)}] Fetching metadata for {repo_name}...")

    api_url = f"https://api.github.com/repos/{username}/{repo}"
    try:
        r = requests.get(api_url, headers=HEADERS)
        if r.status_code == 403:
            print("⏳ Rate limit hit, sleeping for 60 seconds...")
            time.sleep(60)
            continue
        if not r.ok:
            print(f"⚠️ Failed to fetch repo: {repo_name} — {r.status_code}")
            continue

        data = r.json()
        size = data.get("size", 0)
        forks = data.get("forks_count", 0)
        stars = data.get("stargazers_count", 0)
        pushed = data.get("pushed_at", "")
        default_branch = data.get("default_branch", "main")

        # Pull requests
        pulls_resp = requests.get(f"{api_url}/pulls?state=all&per_page=1", headers=HEADERS)
        pulls = int(pulls_resp.headers.get('Link', '').split('&page=')[-1].split('>')[0]) if 'Link' in pulls_resp.headers else len(pulls_resp.json())

        # Contributors
        contribs_resp = requests.get(f"{api_url}/contributors?per_page=1&anon=true", headers=HEADERS)
        contributors = int(contribs_resp.headers.get('Link', '').split('&page=')[-1].split('>')[0]) if 'Link' in contribs_resp.headers else len(contribs_resp.json())

        # Approximate commits (via repo stats)
        commit_api = f"{api_url}/commits?per_page=1&sha={default_branch}"
        commits_resp = requests.get(commit_api, headers=HEADERS)
        commits = int(commits_resp.headers.get('Link', '').split('&page=')[-1].split('>')[0]) if 'Link' in commits_resp.headers else len(commits_resp.json())

        project_metadata.append({
            "project": repo_name,
            "commits": commits,
            "pull_requests": pulls,
            "contributors": contributors,
            "Size": size,
            "Repo_Forks": forks,
            "Repo_Stars": stars,
            "Last_Commit_Date": pushed
        })

    except Exception as e:
        print(f"❌ Error fetching metadata for {repo_name}: {e}")
        continue

# === Save results ===
pd.DataFrame(project_metadata).to_csv(output_csv, index=False)
print(f"\n✅ Done. Analyzed {len(project_metadata)} projects.")
print(f"📁 Saved metadata to: {output_csv}")


🔍 [1/1796] Fetching metadata for AChep.15puzzle...
🔍 [2/1796] Fetching metadata for hapramp.1Rramp-Android...
🔍 [3/1796] Fetching metadata for CSID-DGU.2021-1-OSSP2-Barcode-8...
🔍 [4/1796] Fetching metadata for pknu-wap.2022_2_WAP_APP_TEAM1...
🔍 [5/1796] Fetching metadata for uberspot.2048-android...
🔍 [6/1796] Fetching metadata for TylerCarberry.2048-Battles...
🔍 [7/1796] Fetching metadata for slartus.4pdaClient-plus...
🔍 [8/1796] Fetching metadata for mansoura-cis.4th_Grade_IS-Master-...
🔍 [9/1796] Fetching metadata for felipecsl.6502Android...
🔍 [10/1796] Fetching metadata for mash-up-kr.9tique-android...
🔍 [11/1796] Fetching metadata for patri9ck.a2ln-app...
🔍 [12/1796] Fetching metadata for divVerent.aaaaxy...
🔍 [13/1796] Fetching metadata for atsushieno.aap-juce-frequalizer...
🔍 [14/1796] Fetching metadata for AsyncAlgoTrading.aat...
🔍 [15/1796] Fetching metadata for greenaddress.abcore...
🔍 [16/1796] Fetching metadata for springload.react-accessible-accordion...
🔍 [17/1796] Fetc